In [2]:
pip install pymoo

Note: you may need to restart the kernel to use updated packages.


In [5]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import random
from collections import defaultdict
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import PermutationRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.termination import get_termination
from tqdm import trange
import copy

# Drone specifications (same as in your initial code)
drone_spec = {
    '103': {'drone_name': "Freefly Alta X", 'fly_time': 20 / 60, 'Payload': 15, 'Speed': 25 * 3.6, 'Battery_mAh': 20000, 'Energy_meter': 7, 'Cost_per_energy': 3},
    '104': {'drone_name': "JOUAV CW-80E", 'fly_time': 20 / 60, 'Payload': 10, 'Speed': 20 * 3.6, 'Battery_mAh': 15000, 'Energy_meter': 6.5, 'Cost_per_energy': 2.78},
    '105': {'drone_name': "Draganfly Heavy Lift Drone", 'fly_time': 23 / 60, 'Payload': 30, 'Speed': 17 * 3.6, 'Battery_mAh': 27000, 'Energy_meter': 10, 'Cost_per_energy': 4.2},
    '106': {'drone_name': "DJI FlyCart", 'fly_time': 30 / 60, 'Payload': 18, 'Speed': 20 * 3.6, 'Battery_mAh': 23000, 'Energy_meter': 8, 'Cost_per_energy': 3.4},
    '107': {'drone_name': "Harris Aerial Carrier H6 HL", 'fly_time': 20 / 60, 'Payload': 40, 'Speed': 15 * 3.6, 'Battery_mAh': 30000, 'Energy_meter': 11, 'Cost_per_energy': 4.7}
}

# Scaling factor and location data
scaling_factor = 0.1

location = [
    (6.75859879717907 * scaling_factor, 24.1011220595671 * scaling_factor),
    (64.2020211216308 * scaling_factor, 21.2780005223912 * scaling_factor),
    (3.28223545665507 * scaling_factor, 58.3280585995255 * scaling_factor),
    (0.356408133450226 * scaling_factor, 71.2535984441914 * scaling_factor),
    (47.5416279589111 * scaling_factor, 57.2421947321968 * scaling_factor),
    (54.1391723668124 * scaling_factor, 10.1630144708565 * scaling_factor),
    (5.16311785475916 * scaling_factor, 18.3713689888664 * scaling_factor),
    (58.1077341232713 * scaling_factor, 97.325872003479 * scaling_factor),
    (22.3719171429327 * scaling_factor, 94.5307472331531 * scaling_factor),
    (10.9215147296463 * scaling_factor, 26.9026260431418 * scaling_factor),
    (40.2830425950062 * scaling_factor, 49.0059611185157 * scaling_factor),
    (95.9067950103118 * scaling_factor, 31.2537921390593 * scaling_factor),
    (38.8068853736467 * scaling_factor, 61.8496341187066 * scaling_factor),
    (50.7736281603872 * scaling_factor, 41.7122988413166 * scaling_factor),
    (16.1210451723109 * scaling_factor, 4.17415211185698 * scaling_factor),
    (30.5725260368337 * scaling_factor, 25.6641613944403 * scaling_factor),
    (88.6541848293903 * scaling_factor, 28.6034058450262 * scaling_factor),
    (76.1583329812348 * scaling_factor, 52.0601020081577 * scaling_factor),
    (57.9907902884404 * scaling_factor, 58.2901693224847 * scaling_factor),
    (80.1180100459334 * scaling_factor, 80.7424210060046 * scaling_factor),
    (99.879027610417 * scaling_factor, 99.9769056531731 * scaling_factor),
    (2.58720752188314 * scaling_factor, 45.8084939684228 * scaling_factor),
    (74.0498005304951 * scaling_factor, 2.98613115772289 * scaling_factor),
]

warehouse_locations = [
    (71.7091166549543 * scaling_factor, 17.2581705831028 * scaling_factor),
    (16.6066387616466 * scaling_factor, 73.6752906752222 * scaling_factor),
]

loc_wght = [
    1.5, 2.5, 2.5, 2.0, 2.0, 3.0, 1.0, 3.0, 0.5, 3.5, 1.0, 2.0, 0.0, 3.0, 0.0, 2.0, 0.0, 1.5, 1.0, 2.5, 0.5, 2.0, 1.5
]

def distance_between_locations(p1, p2):
    """Calculate distance between two given coordinates"""
    return math.sqrt(math.pow(p1[0] - p2[0], 2) + math.pow(p1[1] - p2[1], 2))

# The WarehouseManager class
class WarehouseManager:
    def __init__(self):
        self.drone_spec = drone_spec
        self.routes = []
        self.drone_choices = []
        self.route_details = []
        self.deli_locations_short_list = copy.deepcopy(location)

class WarehouseManager:
    def __init__(self):
        self.drone_spec = drone_spec
        self.routes = []
        self.drone_choices = []
        self.route_details = []
        self.deli_locations_short_list = []

    def reset_deli_locationss(self):
        self.deli_locations_short_list = copy.deepcopy(location)

    def find_initial_routes(self):
        max_attempts = 1000
        attempts = 0

        self.reset_deli_locationss()

        while self.deli_locations_short_list and attempts < max_attempts:
            attempts += 1
            warehouse_choice = random.choice(warehouse_locations)
            drone_id = random.choice(list(self.drone_spec.keys()))
            drone_choice = self.drone_spec[drone_id]

            route = [warehouse_choice]
            current_payload = 0
            current_time = 0
            current_location = warehouse_choice

            # Shuffle the deli_locations list aggressively for diversity
            random.shuffle(self.deli_locations_short_list)
            selected_deli_locationss = []

            for deli_locations in self.deli_locations_short_list[:]:
                deli_locations_weight = loc_wght[location.index(deli_locations)]
                dist_to_deli_locations = distance_between_locations(current_location, deli_locations)
                time_to_deli_locations = dist_to_deli_locations / drone_choice['Speed']

                if (current_payload + deli_locations_weight <= drone_choice['Payload'] and
                        current_time + time_to_deli_locations + distance_between_locations(deli_locations, warehouse_choice) / drone_choice['Speed'] <= drone_choice['fly_time']):
                    current_payload += deli_locations_weight
                    current_time += time_to_deli_locations
                    route.append(deli_locations)
                    current_location = deli_locations
                    selected_deli_locationss.append(deli_locations)
                else:
                    if random.random() < 0.5:  # Increase chance of breaking early for diversity
                        break

            return_to_warehouse_time = distance_between_locations(current_location, warehouse_choice) / drone_choice['Speed']
            current_time += return_to_warehouse_time

            if current_time <= drone_choice['fly_time']:
                route.append(warehouse_choice)
                self.routes.append(route)
                self.drone_choices.append(drone_choice['drone_name'])

                self.route_details.append({
                    'drone_id': drone_id,
                    'drone_name': drone_choice['drone_name'],
                    'route': route,
                    'total_weight': current_payload,
                    'total_time': current_time * 60
                })

                # Remove the deli_locationss that were successfully visited in this route
                for deli_locations in selected_deli_locationss:
                    self.deli_locations_short_list.remove(deli_locations)

        if attempts >= max_attempts:
            print("Warning: Reached maximum attempts while finding initial routes.")
        else:
            print(f"Initial routes found in {attempts} attempts.")

    def plot_route(self, route, title, route_id):
        try:
            colours = list(plt.cm.get_cmap('tab10').colors)
            xs, ys = zip(*route)
            plt.figure(figsize=(8, 6))
            plt.plot(xs, ys, color=colours[route_id % len(colours)], marker='o')
            plt.scatter(xs[1:-1], ys[1:-1], color=colours[route_id % len(colours)], s=50)
            plt.scatter(*zip(*location), color='blue', s=100, label='deli_locationss')
            plt.scatter(*zip(*warehouse_locations), color='red', s=200, label='Warehouses', marker='s')
            plt.title(title)
            plt.xlabel('X coordinate')
            plt.ylabel('Y coordinate')
            plt.grid(True)
            plt.show()
            print(f"Route {route_id + 1} plotted successfully.")
        except Exception as e:
            print(f"Failed to plot route {route_id + 1}: {e}")

    def plot_all_routes(self, routes, title):
        """Plot all routes on a single graph."""
        colours = list(mcolors.TABLEAU_COLORS.keys())
        plt.figure(figsize=(10, 8))

        for idx, route in enumerate(routes):
            xs, ys = zip(*route)
            plt.plot(xs, ys, color=colours[idx % len(colours)], marker='o', label=f'Route {idx + 1}')
            plt.scatter(xs[1:-1], ys[1:-1], color=colours[idx % len(colours)], s=50)

        plt.scatter(*zip(*location), color='blue', s=100, label='deli_locationss')
        plt.scatter(*zip(*warehouse_locations), color='red', s=200, label='Warehouses', marker='s')
        plt.title(title)
        plt.xlabel('X coordinate')
        plt.ylabel('Y coordinate')
        plt.grid(True)
        plt.legend(loc='best')
        plt.show()


# The optimization problem class
class DroneDeliveryProblem(Problem):
    def __init__(self, initial_route, drone_spec):
        super().__init__(n_var=len(initial_route) - 2, n_obj=2, n_constr=2, xl=0, xu=1)
        self.initial_route = initial_route
        self.drone_spec = drone_spec

    def _evaluate(self, x, out, *args, **kwargs):
        F1 = np.zeros(x.shape[0])  # Energy cost
        F2 = np.zeros(x.shape[0])  # Delivery time
        g1 = np.zeros(x.shape[0])  # Fly time constraint
        g2 = np.zeros(x.shape[0])  # Payload constraint

        for i in range(x.shape[0]):
            permutation = np.argsort(x[i])
            route = [self.initial_route[0]] + [self.initial_route[j + 1] for j in permutation] + [self.initial_route[-1]]
            drone = self.drone_spec

            total_energy_cost = 0
            total_time = 0
            fly_time_violation = 0
            payload_violation = 0
            current_payload = sum([loc_wght[location.index(loc)] for loc in route[1:-1]])

            for j in range(len(route) - 1):
                src = route[j]
                dst = route[j + 1]
                dist_km = distance_between_locations(src, dst)
                total_distance = dist_km

                energy_used = dist_km * drone['Energy_meter'] * current_payload
                total_energy_cost += energy_used

                if dst in location:
                    delivery_weight = loc_wght[location.index(dst)]
                    current_payload -= delivery_weight

            total_time = total_distance / drone['Speed']
            fly_time_violation = max(0, total_time - drone['fly_time'])
            payload_violation = max(0, current_payload - drone['Payload'])

            F1[i] = total_energy_cost
            F2[i] = total_time
            g1[i] = fly_time_violation
            g2[i] = payload_violation

        out["F"] = np.column_stack([F1, F2])
        out["G"] = np.column_stack([g1, g2])

# The optimizer function
def optimize_route(initial_route, drone):
    problem = DroneDeliveryProblem(initial_route, drone)

    algorithm = NSGA2(
        pop_size=500,
        sampling=PermutationRandomSampling(),
        crossover=SBX(prob=0.9, eta=15),
        mutation=PM(eta=20),
        eliminate_duplicates=True
    )

    termination = get_termination("n_gen", 500)

    try:
        res = minimize(problem, algorithm, termination, seed=1, save_history=True, verbose=False)
    except Exception as e:
        print(f"Optimization failed: {e}")
        return initial_route, (None, None)

    if res.F is None:
        print("Optimization returned no valid results.")
        return initial_route, (None, None)

    best_individual_idx = np.argmin(res.F[:, 0] + res.F[:, 1])
    best_permutation = np.argsort(res.X[best_individual_idx])
    optimized_route = [initial_route[0]] + [initial_route[j + 1] for j in best_permutation] + [initial_route[-1]]

    return optimized_route, res.F[best_individual_idx]

# Calculate total distance and cost
def calculate_total_distance_and_cost(routes, drone_choices):
    total_distance = 0
    total_cost = 0
    total_time = 0

    for route, drone_name in zip(routes, drone_choices):
        drone_details = next(item for item in drone_spec.values() if item['drone_name'] == drone_name)
        current_payload = sum([loc_wght[location.index(loc)] for loc in route[1:-1]])
        route_distance = 0
        route_cost = 0
        route_time = 0

        for i in range(len(route) - 1):
            src = route[i]
            dst = route[i + 1]
            dist_km = distance_between_locations(src, dst)
            route_distance += dist_km
            energy_used = dist_km * drone_details['Energy_meter'] * current_payload
            segment_cost = energy_used * drone_details['Cost_per_energy']
            route_cost += segment_cost
            route_time += dist_km / drone_details['Speed']
            if dst in location:
                delivery_weight = loc_wght[location.index(dst)]
                current_payload -= delivery_weight

        total_distance += route_distance
        total_cost += route_cost
        total_time += route_time

    return total_time, total_cost

def perturb_route(route):
    if len(route) > 3:
        i, j = random.sample(range(1, len(route) - 1), 2)
        route[i], route[j] = route[j], route[i]
    return route

# Hill Climbing with TQDM
from tqdm import tqdm

def generate_initial_routes_hill_climbing(n):
    best_initial_routes = None
    best_optimized_routes = None
    best_cost = float('inf')
    best_time = float('inf')
    best_wm_instance = None
    results = []

    for i in tqdm(range(n), desc="Generating and optimizing routes"):
        random.seed()  # Optional: Reset random seed for diversity

        wm = WarehouseManager()
        wm.find_initial_routes()

        tqdm.write(f"Initial routes found in {len(wm.routes)} attempts.")

        # Calculate total time and cost for the initial routes
        initial_total_time, initial_total_cost = calculate_total_distance_and_cost(wm.routes, wm.drone_choices)

        optimized_routes = []
        for idx, initial_route in enumerate(wm.routes):
            drone_id = wm.route_details[idx]['drone_id']
            drone = wm.drone_spec[drone_id]

            # Optimize the route
            optimized_route, fitness = optimize_route(initial_route, drone)
            if isinstance(fitness, (list, np.ndarray)) and not np.any(np.isnan(fitness)):
                optimized_routes.append(optimized_route)
            else:
                optimized_routes.append(initial_route)


        # Calculate total time and cost for the optimized routes
        optimized_total_time, optimized_total_cost = calculate_total_distance_and_cost(optimized_routes, wm.drone_choices)

        results.append((initial_total_time, initial_total_cost, optimized_total_time, optimized_total_cost, wm.routes, optimized_routes, wm))

        if optimized_total_cost < best_cost and optimized_total_time < best_time: #(optimized_total_cost == best_cost and optimized_total_time < best_time):
            best_initial_routes = wm.routes
            best_optimized_routes = optimized_routes
            best_cost = optimized_total_cost
            best_time = optimized_total_time
            best_wm_instance = copy.deepcopy(wm)

    return best_initial_routes, best_optimized_routes, best_cost, best_time, best_wm_instance, results



# Generate and compare 100 different route sets


best_initial_routes, best_optimized_routes, best_cost, best_time, best_wm_instance, results = generate_initial_routes_hill_climbing(100)

# Print total flight time and energy cost for initial and optimized routes
for idx, (initial_time, initial_cost, optimized_time, optimized_cost, initial_routes, optimized_routes, wm) in enumerate(results):
    print(f"Route Set {idx + 1}:")
    print(f"  Initial Total Cost: ${initial_cost:.2f}, Initial Total Time: {initial_time:.2f} hrs")
    print(f"  Optimized Total Cost: ${optimized_cost:.2f}, Optimized Total Time: {optimized_time:.2f} hrs")
    print(f"Total Optimized Cost for Route Set {idx + 1}: ${optimized_cost:.2f}, Total Time: {optimized_time:.2f} hrs\n")

print(f"\nBest Route Set found with Cost: ${best_cost:.2f} and Time: {best_time:.2f} hrs")

# Plot each best initial and optimized route individually
for idx, (initial_route, optimized_route) in enumerate(zip(best_initial_routes, best_optimized_routes)):
    best_wm_instance.plot_route(initial_route, f"Best Initial Route {idx + 1}", idx)
    best_wm_instance.plot_route(optimized_route, f"Best Optimized Route {idx + 1}", idx + len(best_initial_routes))

# Plot all best initial routes in one graph
best_wm_instance.plot_all_routes(best_initial_routes, "All Best Initial Routes")

# Plot all best optimized routes in another graph
best_wm_instance.plot_all_routes(best_optimized_routes, "All Best Optimized Routes")





Generating and optimizing routes:   0%|          | 0/100 [00:00<?, ?it/s]

Initial routes found in 7 attempts.
Initial routes found in 7 attempts.


Generating and optimizing routes:   1%|          | 1/100 [00:04<07:15,  4.40s/it]

Initial routes found in 6 attempts.
Initial routes found in 6 attempts.


Generating and optimizing routes:   2%|▏         | 2/100 [00:08<06:27,  3.95s/it]

...

Initial routes found in 7 attempts.
Initial routes found in 7 attempts.


Generating and optimizing routes:  11%|█         | 11/100 [00:50<06:58,  4.70s/it]

Initial routes found in 8 attempts.
Initial routes found in 8 attempts.


Generating and optimizing routes:  11%|█         | 11/100 [00:53<07:13,  4.88s/it]


KeyboardInterrupt: 